In [0]:
import time
from pyspark.sql import functions as F

# 强行引入你的真大表
oi_source = spark.table("bronze_order_items")

# ==========================================
# 🚀 1. 基准测试（Baseline）：体验未经优化的地狱重算
# ==========================================
print("=== 🏃 正在死算基准测试（Baseline）... ===")
start_base = time.time()

# 模拟未优化的传统单层聚合
df_baseline = oi_source.groupBy("seller_id").agg(F.count("*").alias("total_orders"))
df_baseline.write.format("delta").mode("overwrite").saveAsTable("yuto_report_baseline")

time_base = time.time() - start_base

# 🎉 修正点 1：使用标准的 DESCRIBE DETAIL 语句，并用 select 抓取状态
base_detail = spark.sql("DESCRIBE DETAIL yuto_report_baseline").select("numFiles", "sizeInBytes").collect()[0]
files_base = base_detail["numFiles"]
io_base = base_detail["sizeInBytes"] / (1024 * 1024) # 转化为 MB

print(f"✅ 基准测试完成！耗时: {time_base:.2f}秒 | 产出文件: {files_base}个 | 磁盘I/O: {io_base:.2f} MB\n")



In [0]:

# ==========================================
# 🚀 2. 调优后测试（Optimized）：加盐分流 + 落盘防线 + Coalesce 终极组合拳
# ==========================================
print("=== 🏃 正在死算黄金调优后测试（Optimized）... ===")
start_opt = time.time()

# 【一阶段】加随机盐
df_salted = oi_source.withColumn("salted_id", F.concat(F.col("seller_id"), F.lit("_"), F.floor(F.rand() * 10)))
df_stage1 = df_salted.groupBy("salted_id").agg(F.count("*").alias("partial_count"))

# 【二阶段】去盐汇总
df_desalted = df_stage1.withColumn("real_id", F.split(F.col("salted_id"), "_")[0])
df_stage2 = df_desalted.groupBy("real_id").agg(F.sum("partial_count").alias("total_orders"))

# 【三阶段】coalesce(1) 终极瘦身落盘
df_final_optimized = df_stage2
df_final_optimized.write.format("delta").mode("overwrite").saveAsTable("yuto_report_optimized")

time_opt = time.time() - start_opt

# 🎉 修正点 2：同样改用 DESCRIBE DETAIL
opt_detail = spark.sql("DESCRIBE DETAIL yuto_report_optimized").select("numFiles", "sizeInBytes").collect()[0]
files_opt = opt_detail["numFiles"]
io_opt = opt_detail["sizeInBytes"] / (1024 * 1024)

print(f"✅ 调优后测试完成！耗时: {time_opt:.2f}秒 | 产出文件: {files_opt}个 | 磁盘I/O: {io_opt:.2f} MB")

可以发现加盐了之后还是比第一个速度要快的

然后我们再测试一下，假设在加盐法的stage2后加一个cache（1），强行合并成一个机器执行，会不会增加效率

In [0]:

# ==========================================
# 🚀 2. 调优后测试（Optimized）：加盐分流 + 落盘防线 + Coalesce 终极组合拳
# ==========================================
print("=== 🏃 正在死算黄金调优后测试（Optimized）... ===")
start_opt = time.time()

# 【一阶段】加随机盐
df_salted = oi_source.withColumn("salted_id", F.concat(F.col("seller_id"), F.lit("_"), F.floor(F.rand() * 10)))
df_stage1 = df_salted.groupBy("salted_id").agg(F.count("*").alias("partial_count"))

# 【二阶段】去盐汇总
df_desalted = df_stage1.withColumn("real_id", F.split(F.col("salted_id"), "_")[0])
df_stage2 = df_desalted.groupBy("real_id").agg(F.sum("partial_count").alias("total_orders"))

# 【三阶段】coalesce(1) 终极瘦身落盘
df_final_optimized = df_stage2.coalesce(1)
df_final_optimized.write.format("delta").mode("overwrite").saveAsTable("yuto_report_optimized")

time_opt = time.time() - start_opt

# 🎉 修正点 2：同样改用 DESCRIBE DETAIL
opt_detail = spark.sql("DESCRIBE DETAIL yuto_report_optimized").select("numFiles", "sizeInBytes").collect()[0]
files_opt = opt_detail["numFiles"]
io_opt = opt_detail["sizeInBytes"] / (1024 * 1024)

print(f"✅ 调优后测试完成！耗时: {time_opt:.2f}秒 | 产出文件: {files_opt}个 | 磁盘I/O: {io_opt:.2f} MB")